## Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import trim, col
from pyspark.sql.window import Window

## Read bronze table

In [0]:
df = spark.table("workspace.bronze.crm_prd_info")

## Silver Transformation

### Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

### Product key parsing

In [0]:
df = df.withColumn("cat_id", F.regexp_replace(F.substring(col("prd_key"), 1, 5), "-", "_"))
df = df.withColumn("prd_key", F.substring(col("prd_key"), 7, F.length(col("prd_key"))))

### Product cost cleanup

In [0]:
df = df.withColumn("prd_cost", F.coalesce(col("prd_cost"), F.lit(0)))

### Product line normalization

In [0]:
df = (
    df
    .withColumn(
        "prd_line",
        F.when(trim(F.upper(col("prd_line"))) == "R", "Road")
        .when(trim(F.upper(col("prd_line"))) == "M", "Mountain")
        .when(trim(F.upper(col("prd_line"))) == "T", "Touring")
        .when(trim(F.upper(col("prd_line"))) == "S", "Other Sales")
        .otherwise("n/a")
    )
)


### Start date casting

In [0]:
df = df.withColumn("prd_start_dt", col("prd_start_dt").cast(DateType()))

### Calculate end date 

In [0]:
window_spec = Window.partitionBy("prd_key").orderBy("prd_start_dt")

df = df.withColumn("prd_end_dt", F.date_sub(F.lead(col("prd_start_dt")).over(window_spec), 1))

### Renaming columns

In [0]:
RENAME_MAP = {
    "prd_id": "product_id",
    "cat_id": "category_id",
    "prd_key": "product_key",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date"   
}

for old, new in RENAME_MAP.items():
    df = df.withColumnRenamed(old, new)


### Reorder columns

In [0]:
df = df.select(
    "product_id",
    "category_id",
    "product_key",
    "product_name",
    "product_cost",
    "product_line",
    "start_date",
    "end_date"
)

### Sanity check columns

In [0]:
df.limit(30).display()

product_id,category_id,product_key,product_name,product_cost,product_line,start_date,end_date
601,CO_BB,BB-7421,LL Bottom Bracket,24,n/a,2013-07-01,null
602,CO_BB,BB-8107,ML Bottom Bracket,45,n/a,2013-07-01,null
603,CO_BB,BB-9108,HL Bottom Bracket,54,n/a,2013-07-01,null
478,AC_BC,BC-M005,Mountain Bottle Cage,4,Mountain,2013-07-01,null
479,AC_BC,BC-R205,Road Bottle Cage,3,Road,2013-07-01,null
596,BI_MB,BK-M18B-40,Mountain-500 Black- 40,295,Mountain,2013-07-01,null
597,BI_MB,BK-M18B-42,Mountain-500 Black- 42,295,Mountain,2013-07-01,null
598,BI_MB,BK-M18B-44,Mountain-500 Black- 44,295,Mountain,2013-07-01,null
599,BI_MB,BK-M18B-48,Mountain-500 Black- 48,295,Mountain,2013-07-01,null
600,BI_MB,BK-M18B-52,Mountain-500 Black- 52,295,Mountain,2013-07-01,null


### Writing silver table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_products")

### Sanity check silver table

In [0]:
%sql
SELECT * FROM workspace.silver.crm_products LIMIT 30

product_id,category_id,product_key,product_name,product_cost,product_line,start_date,end_date
601,CO_BB,BB-7421,LL Bottom Bracket,24,n/a,2013-07-01,null
602,CO_BB,BB-8107,ML Bottom Bracket,45,n/a,2013-07-01,null
603,CO_BB,BB-9108,HL Bottom Bracket,54,n/a,2013-07-01,null
478,AC_BC,BC-M005,Mountain Bottle Cage,4,Mountain,2013-07-01,null
479,AC_BC,BC-R205,Road Bottle Cage,3,Road,2013-07-01,null
596,BI_MB,BK-M18B-40,Mountain-500 Black- 40,295,Mountain,2013-07-01,null
597,BI_MB,BK-M18B-42,Mountain-500 Black- 42,295,Mountain,2013-07-01,null
598,BI_MB,BK-M18B-44,Mountain-500 Black- 44,295,Mountain,2013-07-01,null
599,BI_MB,BK-M18B-48,Mountain-500 Black- 48,295,Mountain,2013-07-01,null
600,BI_MB,BK-M18B-52,Mountain-500 Black- 52,295,Mountain,2013-07-01,null
